# Croissant Baker on Cybench

The Ireland maize folder is one dataset of nine CSVs, the same folder the Cybench ReAct notebook uses. [Croissant Baker](https://github.com/MIT-LCP/croissant-baker) reads those files locally and writes validated Croissant JSON-LD. It does not call a model.

The result is `outputs/croissant_baker_cybench/cybench_IE.jsonld`. An existing file is kept and not replaced.

Install the package in this kernel if the setup cell reports a missing import: `pip install croissant-baker`. Run the setup cell, then the bake cell. After a kernel restart, rerun the setup cell first.

In [ ]:
import json
from pathlib import Path

BASE = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/config.py').is_file()),
    None,
)
if BASE is None:
    raise FileNotFoundError('Could not locate the metadata_agent project root')

try:
    from croissant_baker.metadata_generator import MetadataGenerator
except ImportError as exc:
    raise ImportError('Install croissant-baker in this kernel: pip install croissant-baker') from exc

CYBENCH_DIR = BASE / 'data/experiment/cybench-maize-IE/IE'
OUT = BASE / 'outputs/croissant_baker_cybench'
JSONLD_PATH = OUT / 'cybench_IE.jsonld'
REPORT_PATH = OUT / 'cybench_IE_scan_report.json'
EXPECTED_FILES = 9
DATASET_NAME = 'cybench_IE'
DESCRIPTION = (
    'Cybench Ireland maize benchmark: crop calendar, crop mask, fAPAR, location, '
    'meteorology, NDVI, soil, soil moisture, and yield tables.'
)

csv_files = sorted(CYBENCH_DIR.glob('*.csv'))
if len(csv_files) != EXPECTED_FILES:
    raise ValueError(f'Expected {EXPECTED_FILES} Cybench CSV files, found {len(csv_files)}')
OUT.mkdir(parents=True, exist_ok=True)

print('Project:', BASE)
print('Cybench:', CYBENCH_DIR)
print('CSV files:', len(csv_files))
for path in csv_files:
    print(' ', path.name)
print('Output:', JSONLD_PATH)
print('Existing result kept:', JSONLD_PATH.is_file())


In [ ]:
if JSONLD_PATH.is_file():
    print('Keeping existing Croissant file:', JSONLD_PATH)
else:
    generator = MetadataGenerator(
        dataset_path=str(CYBENCH_DIR),
        name=DATASET_NAME,
        description=DESCRIPTION,
        count_csv_rows=True,
        detect_references=True,
    )
    generator.save_metadata(str(JSONLD_PATH), validate=True)
    report = generator.scan_report
    REPORT_PATH.write_text(json.dumps(report.to_dict(), indent=2), encoding='utf-8')
    print('Wrote', JSONLD_PATH)
    print('Scan report:', REPORT_PATH)
    for line in report.summary_lines():
        print(line)
    references = generator.reference_report
    if references is not None:
        for line in references.summary_lines(verbose=True):
            print(line)
